In [1]:
import os
os.chdir('../../..')

In [2]:
import selfies as sf
from scipy.spatial.distance import cosine, euclidean
from src.datasets import QM9Dataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import polars as pl
from rdkit import DataStructs, Chem
from rdkit.Chem import AllChem
from src.clusters import ClusterAnalysis

In [3]:
qm9 = QM9Dataset()
qm9.load()

2026-03-16 08:57:06.660 | INFO     | src.datasets:load:109 - Loading QM9 from data/QM9/dataset_cleaned.csv...


mol_id,smiles,canonical_smiles,selfies,num_atoms,structure_class,mol_weight,logp,tpsa,num_heavy_atoms,num_rings,num_aromatic_rings,num_rotatable_bonds,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C
str,str,str,str,i64,str,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""qm9_0""","""[H]C([H])([H])[H]""","""[H]C([H])([H])[H]""","""[H][C][Branch1][C][H][Branch1]…",5,"""Acyclic""",16,0,0,1,0,0,0,1.0,0,0,1,0,0,1,2,9,0,0,0,0,0,0,0,0,0,0,0,0.0,13.21,-10.549854,3.186453,13.736308,35.364101,1.217682,-1101.487793,-1101.40979,-1101.384033,-1102.022949,6.469,-17.172182,-17.286823,-17.389656,-16.151918,157.711807,157.709976,157.706985
"""qm9_1""","""[H]N([H])[H]""","""[H]N([H])[H]""","""[H][N][Branch1][C][H][H]""",4,"""Acyclic""",17,0,35,1,0,0,0,0.0,1,1,1,0,0,0,2,6,0,0,0,0,0,0,0,0,0,0,1,1.6256,9.46,-6.993326,2.255824,9.249149,26.1563,0.934929,-1538.147705,-1538.069824,-1538.044189,-1538.666748,6.316,-12.005855,-12.082129,-12.159273,-11.246005,293.609741,293.541107,191.393967
"""qm9_2""","""[H]O[H]""","""[H]O[H]""","""[H][O][H]""",3,"""Acyclic""",18,0,31,1,0,0,0,0.0,0,0,0,0,0,0,2,3,0,0,0,0,0,0,0,0,0,0,1,1.8511,6.31,-7.967494,1.869422,9.836916,19.0002,0.581643,-2079.077881,-2079.000732,-2078.975098,-2079.558105,6.002,-9.240362,-9.278811,-9.330214,-8.733849,799.588135,437.90387,282.945465
"""qm9_3""","""[H]C#C[H]""","""[H]C#C[H]""","""[H][C][#C][H]""",4,"""Acyclic""",26,0,0,2,0,0,0,0.0,0,0,0,2,0,0,3,4,0,0,0,0,0,0,0,0,0,0,0,0.0,16.280001,-7.741639,1.376896,9.118535,59.524799,0.730381,-2103.669434,-2103.590576,-2103.564697,-2104.186523,8.574,-16.716963,-16.792231,-16.869347,-15.862634,0.0,35.610035,35.610035
"""qm9_4""","""[H]C#N""","""[H]C#N""","""[H][C][#N]""",3,"""Acyclic""",27,0,23,2,0,0,0,0.0,0,1,0,1,0,0,2,3,0,0,0,0,0,0,0,0,0,0,1,2.8937,12.99,-9.806983,0.519737,10.329442,48.747601,0.451736,-2541.866943,-2541.79834,-2541.772705,-2542.393555,6.278,-13.088188,-13.13529,-13.186666,-12.520096,0.0,44.593884,44.593884
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""qm9_2230""","""[H]C(=O)N([H])C1([H])C([H])([H…","""[H]C(=O)N([H])C1([H])C([H])([H…","""[H][C][=Branch1][C][=O][N][Bra…",14,"""Aliphatic Ring""",101,0,38,7,1,0,1,0.75,1,2,5,0,1,3,5,28,0,0,0,0,1,0,0,0,1,0,3,2.7698,55.389999,-6.976999,0.536064,7.513064,858.688416,3.099241,-9842.638672,-9842.450195,-9842.423828,-9843.509766,23.656,-57.369255,-57.719849,-58.053978,-53.404961,7.7071,1.43215,1.39873
"""qm9_2231""","""[H]C(=O)OC1([H])C([H])([H])C([…","""[H]C(=O)OC1([H])C([H])([H])C([…","""[H][C][=Branch1][C][=O][O][C][…",15,"""Aliphatic Ring""",100,0,26,7,1,0,1,0.8,0,2,5,0,1,4,6,31,0,0,0,0,0,0,0,0,1,0,2,4.0416,59.34,-7.518506,0.274835,7.793341,940.18573,3.398049,-9406.087891,-9405.898438,-9405.873047,-9406.954102,24.367001,-62.644833,-63.03352,-63.393368,-58.398766,9.30716,1.3292,1.24484
"""qm9_2232""","""[H]C(=O)OC1([H])C([H])([H])OC1…","""[H]C(=O)OC1([H])C([H])([H])OC1…","""[H][C][=Branch1][C][=O][O][C][…",13,"""Aliphatic Ring""",102,0,35,7,1,0,1,0.75,0,3,4,0,1,3,5,25,0,0,0,0,0,0,0,0,2,0,3,2.762,51.860001,-7.235507,-0.059865,7.175642,893.72937,2.746336,-10383.348633,-10383.164062,-10383.138672,-10384.21582,22.610001,-54.383949,-54.699978,-55.008392,-50.708042,9.96411,1.33561,1.25859
